# Phase 5 — Evaluation
## Brain Tumour MRI Classification
====================================================================

The test set is opened here for the first time.

Every headline number is reported beside the two stratifications that qualify
it: by native file size, because the audit in Phase 1 proved a source shortcut
exists, and by similarity to the training set, because patient identity cannot
be verified on this dataset.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from src import (config, data, engine, explain, manifest, metrics, robustness,
                 splits, viz)
from src.config import (CACHE_DIR, CKPT_PATH, CLASSES, DEVICE, FACE, PALETTE,
                        TEST_DIR, TRAIN_DIR)

train_img, train_lab, _, test_img, test_lab, test_paths, C = \
    splits.build_caches(TRAIN_DIR, TEST_DIR)
S = splits.build_splits(train_img, train_lab, test_img)
KEEP = ~S["test_leak"]

# everything downstream sees only the cleaned test set
test_img, test_lab = test_img[KEEP], test_lab[KEEP]
test_paths = [p for p, k in zip(test_paths, KEEP) if k]

model, ckpt = engine.load_checkpoint(CKPT_PATH)
MEAN, STD = ckpt["norm"]
IMG = ckpt["img_size"]

# A figure and a checkpoint from different runs once sat in an outputs folder
# looking equally authoritative. The manifest makes that impossible to miss.
assert ckpt.get("manifest") == manifest.current_hash(), (
    f"checkpoint {ckpt.get('manifest')} does not match manifest "
    f"{manifest.current_hash()} -- re-run Phases 1 and 3")

print(f"checkpoint  epoch {ckpt['epoch']}  val acc {ckpt['val_acc']:.4f}  "
      f"deep={ckpt['deep']}  act={ckpt['activation']}")
print(f"preprocessing travelled with it: img_size {IMG}, MEAN {MEAN:.4f}, STD {STD:.4f}")
print(f"run {manifest.current_hash()} -- checkpoint and figures agree")

checkpoint  epoch 53  val acc 0.9744  deep=True  act=relu
preprocessing travelled with it: img_size 128, MEAN 0.2200, STD 0.1840
run 0ec49f71495d -- checkpoint and figures agree


In [2]:
# 1. OPENING THE TEST SET, ONCE
"""
This is the first time these images are used for anything. They were not
involved in choosing the architecture, the learning rate, the augmentation, the
stopping epoch, the balancing strategy or the normalisation constants -- all of
that was decided against the validation split in Phases 3 and 4.

That distinction is the whole value of the number below. A test set that has
been peeked at during development stops being a test set and becomes a second
validation set carrying an optimistic bias nobody can measure.

The set scored here is the cleaned one from Phase 1: 470 of the 1,497 images
were dropped as duplicates of, or same-patient near-neighbours of, training
scans. Phase 1 asserted that nothing survived that exclusion, and the assertion
was re-derived independently rather than trusting the mask that produced it.
"""
eval_tf = data.make_transforms(MEAN, STD, augment=False, img_size=IMG)
test_ds = data.CachedDataset(test_img, test_lab, eval_tf)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=32, shuffle=False)

y_true, y_pred, probs = metrics.predict(model, test_loader)
assert len(y_true) == len(test_lab), "loader dropped images -- check drop_last"
print(f"test images scored: {len(y_true)}   "
      f"({(~KEEP).sum()} excluded in Phase 1 as leaked)")
print(f"per class: " + "  ".join(f"{c} {int((y_true==i).sum())}"
                                 for i, c in enumerate(CLASSES)))

test images scored: 1027   (470 excluded in Phase 1 as leaked)
per class: glioma 375  meningioma 190  notumor 88  pituitary 374


In [3]:
# 2. HEADLINE, AND WHY MACRO F1 LEADS
"""
Accuracy is reported because it is expected, not because it is the right
summary here. Cleaning the dataset left the test set uneven -- 88 notumor images
against 375 glioma -- and accuracy on an uneven set is a weighted average
dominated by the large classes. A model that handled the three big classes well
and failed notumor entirely would still post a respectable figure.

Macro F1 weights every class equally, so that failure would show. It is the
number to quote.

Both come with a bootstrap interval, because a figure computed on one particular
sample of a thousand images has real uncertainty attached and quoting four
decimal places implies a precision the sample size does not support.
"""
acc, acc_lo, acc_hi = metrics.bootstrap_ci(y_true, y_pred, "accuracy")
f1, f1_lo, f1_hi = metrics.bootstrap_ci(y_true, y_pred, "macro_f1",
                                        n_classes=len(CLASSES))
print(f"  macro F1   {f1:.4f}   95% CI [{f1_lo:.4f}, {f1_hi:.4f}]   <- the headline")
print(f"  accuracy   {acc:.4f}   95% CI [{acc_lo:.4f}, {acc_hi:.4f}]")
print(f"\n  chance                      {1/len(CLASSES):.4f}")
print(f"  validation (Phase 3)        {ckpt['val_acc']:.4f}   <- selection metric, biased upward")

if acc > 0.995:
    print("\n  WARNING: above 99.5% on this dataset is more likely leakage than skill.")

  macro F1   0.9247   95% CI [0.9053, 0.9431]   <- the headline
  accuracy   0.9367   95% CI [0.9211, 0.9513]

  chance                      0.2500
  validation (Phase 3)        0.9744   <- selection metric, biased upward


In [4]:
# 3. CONFUSION MATRIX AND PER-CLASS SCORES
"""
The confusion matrix answers what accuracy hides: not how many were wrong, but
which class got mistaken for which.

A prediction stated in advance, because a result matching a prediction is worth
more than one explained afterwards: glioma and meningioma should dominate the
confusions. Both are masses presenting as bright regions, and what separates
them is whether the tumour arises inside brain tissue or from the meninges
covering it -- a distinction about margin and position rather than intensity. If
notumor were being confused with anything, that would point at a bug rather than
at difficulty.
"""
cm = metrics.confusion(y_true, y_pred, len(CLASSES))
viz.plot_confusion(cm, CLASSES)

off = sorted(((cm[i, j], CLASSES[i], CLASSES[j])
              for i in range(len(CLASSES)) for j in range(len(CLASSES)) if i != j),
             reverse=True)
print("largest confusions:")
for n, a, b in off[:4]:
    if n:
        print(f"  {n:>4} {a} scans predicted as {b}")

print()
rows = metrics.per_class_report(y_true, y_pred, CLASSES)
metrics.print_report(rows)

print(f"\n{'class':<14}{'recall':>9}{'95% CI':>20}{'support':>9}")
print("-" * 52)
for i, name in enumerate(CLASSES):
    p, lo, hi = metrics.bootstrap_ci(y_true, y_pred, class_idx=i)
    print(f"{name:<14}{p:>9.4f}   [{lo:.4f}, {hi:.4f}]{int((y_true==i).sum()):>9}")

  saved -> outputs/confusion_matrix.png
largest confusions:
    32 glioma scans predicted as meningioma
    20 glioma scans predicted as notumor
     6 meningioma scans predicted as pituitary
     3 pituitary scans predicted as meningioma

class             precision   recall       f1  support
------------------------------------------------------
glioma               0.9938   0.8560   0.9198      375
meningioma           0.8394   0.9632   0.8971      190
notumor              0.8148   1.0000   0.8980       88
pituitary            0.9788   0.9893   0.9840      374
------------------------------------------------------
macro avg            0.9067   0.9521   0.9247     1027
weighted avg         0.9445   0.9367   0.9371     1027

class            recall              95% CI  support
----------------------------------------------------
glioma           0.8560   [0.8187, 0.8907]      375
meningioma       0.9632   [0.9316, 0.9842]      190
notumor          1.0000   [1.0000, 1.0000]       88
pi

In [5]:
# 4. ROC AND AUC
"""
AUC is threshold-free: it measures whether the model ranks positives above
negatives across every operating point, rather than scoring the single argmax
decision. That makes it a different question from accuracy and a more forgiving
one -- a model can rank well and still put its argmax on the wrong class.

One-vs-rest on four classes also means each curve's negative pool is three times
the positive one and is dominated by easy negatives, which inflates the number
further. It is reported because it is conventional, and it should not be the
figure anyone quotes.
"""
curves, aucs, macro_auc = metrics.roc_ovr(y_true, probs, len(CLASSES))
viz.plot_roc(curves, aucs, macro_auc, CLASSES)
for name, a in zip(CLASSES, aucs):
    print(f"  {name:<14}AUC {a:.4f}")
print(f"  {'macro':<14}AUC {macro_auc:.4f}")

  saved -> outputs/roc_curves.png
  glioma        AUC 0.9599
  meningioma    AUC 0.9913
  notumor       AUC 0.9947
  pituitary     AUC 0.9995
  macro         AUC 0.9863


In [6]:
# 5. THE SHORTCUT, STRATIFIED
"""
Phase 1 measured, before any training, that a random forest given only width,
height, file size and bytes-per-pixel -- never a pixel -- classifies these images
at 0.6158 against 0.2500 chance. The cause is structural: the tumour classes
come overwhelmingly from 512x512 source images and notumor almost never does.

That is a property of the data and cannot be preprocessed away. Resizing to
128px hides the dimensions but not the resampling signature, and an augmentation
built specifically to destroy that signature was measured, on this dataset, to
change nothing -- augmentation cannot invent training examples of a class in a
style that occurs zero times under that label.

So the obligation is to report it. If the model leaned on the shortcut,
accuracy collapses on exactly the images where it points the wrong way: tumour
scans that are not 512x512. That is falsifiable, and this is the test.
"""
sizes = np.array([Image.open(p).size for p in test_paths])
is512 = (sizes[:, 0] == 512) & (sizes[:, 1] == 512)
correct = y_true == y_pred

print(f"{'class':<14}{'512x512':>9}{'other':>8}{'acc|512':>10}{'acc|other':>12}{'gap':>9}")
print("-" * 62)
gaps = []
for c, name in enumerate(CLASSES):
    m = y_true == c
    a, b = m & is512, m & ~is512
    av = correct[a].mean() if a.sum() else float('nan')
    bv = correct[b].mean() if b.sum() else float('nan')
    gaps.append(av - bv)
    print(f"{name:<14}{a.sum():>9}{b.sum():>8}{av:>10.3f}{bv:>12.3f}{av-bv:>9.3f}")

print(f"\noverall   512x512 {correct[is512].mean():.4f} (n={is512.sum()})   "
      f"other {correct[~is512].mean():.4f} (n={(~is512).sum()})")

viz.plot_bars(CLASSES + ["overall"],
              [correct[(y_true == c) & ~is512].mean()
               if ((y_true == c) & ~is512).sum() else np.nan
               for c in range(len(CLASSES))] + [correct[~is512].mean()],
              "shortcut_audit.png",
              "Accuracy on scans that are NOT 512x512 — where the shortcut misleads",
              hline=correct.mean())

worst = np.nanmax(gaps)
print(f"\nlargest per-class gap: {worst:.3f}")
print("-> the shortcut is exploited" if worst > 0.20 else
      "-> the shortcut is available in the data but this model does not lean on it")

class           512x512   other   acc|512   acc|other      gap
--------------------------------------------------------------
glioma              313      62     0.974       0.258    0.716
meningioma          180      10     0.967       0.900    0.067
notumor               0      88       nan       1.000      nan
pituitary           366       8     0.989       1.000   -0.011

overall   512x512 0.9790 (n=859)   other 0.7202 (n=168)
  saved -> outputs/shortcut_audit.png

largest per-class gap: 0.716
-> the shortcut is exploited


In [7]:
# 6. RESIDUAL LEAKAGE, STRATIFIED
"""
Phase 1 removed every duplicate and every near-neighbour above cosine 0.95, and
asserted that nothing survived. What it could not remove is same-patient bleed
below that threshold, because this dataset ships no patient identifiers and a
pixel comparison cannot tell a different slice of one patient from a different
patient.

This bounds what is left. Accuracy is split by how similar each test image is to
its nearest training image: if residual leakage were inflating the headline, the
most-similar band would score conspicuously higher than the least-similar one.

The least-similar band is the honest floor, and it belongs beside the headline.
"""
_, _, cos = data.find_duplicates(test_img, train_img[S["train_keep"]])
print(f"{'similarity to nearest training image':<40}{'n':>6}{'accuracy':>11}")
print("-" * 57)
bands = [(0.90, 1.01, "0.90 - 0.95  (most similar)"),
         (0.85, 0.90, "0.85 - 0.90"),
         (0.00, 0.85, "below 0.85   (least similar)")]
for lo, hi, label in bands:
    m = (cos >= lo) & (cos < hi)
    if m.sum():
        print(f"{label:<40}{m.sum():>6}{correct[m].mean():>11.4f}")
print(f"{'ALL':<40}{len(correct):>6}{correct.mean():>11.4f}")

low = cos < 0.85
print(f"\nhonest floor -- accuracy on the {low.sum()} least-similar images: "
      f"{correct[low].mean():.4f}")
print(f"headline accuracy:                                    {correct.mean():.4f}")

similarity to nearest training image         n   accuracy
---------------------------------------------------------
0.90 - 0.95  (most similar)                265     0.9925
0.85 - 0.90                                209     0.9617
below 0.85   (least similar)               553     0.9005
ALL                                       1027     0.9367

honest floor -- accuracy on the 553 least-similar images: 0.9005
headline accuracy:                                    0.9367


In [8]:
# 7. ROBUSTNESS — HOW FAR DOES THE NUMBER TRAVEL?
"""
Every figure so far describes this dataset. The standard objection to a high
accuracy on a curated public MRI benchmark is not that the number is faked, it
is that the benchmark is clean in ways a hospital is not: consistently windowed,
uniformly sharp, low noise. A model can learn to depend on all of that, and no
ordinary evaluation shows the dependence, because the test split is clean in
exactly the same ways.

So the test set is degraded and scored again. This is not external validation
and does not pretend to be. It answers a narrower question honestly: does the
accuracy survive the variation that separates one scanner from another?
"""
rob = robustness.robustness_test(model, test_img, y_true, MEAN, STD, IMG,
                                 n_classes=len(CLASSES))
robustness.print_robustness(rob)

labels = list(rob)
viz.plot_bars(labels, [rob[k]["f1"] for k in labels], "robustness.png",
              "Macro F1 under acquisition variation", ylabel="macro F1",
              hline=rob["clean"]["f1"],
              colours=[PALETTE["good"]] + [PALETTE["val"]] * (len(labels) - 1))

print("""
Read the bias-field row against the noise and blur rows. Bias field is the most
MRI-specific corruption here and a model reading anatomy should barely notice
it; noise and blur disturb high-frequency texture. Which of those hurts more
says something about what the model is using.""")

  condition                 accuracy  macro F1   drop (F1)
  --------------------------------------------------------
  clean                       0.9367    0.9247
  noise sigma=5               0.9338    0.9225     +0.0022
  noise sigma=15              0.7069    0.6956     +0.2291
  blur radius 1px             0.7936    0.8078     +0.1169
  blur radius 2px             0.4752    0.4624     +0.4623
  gamma 0.7                   0.8929    0.8816     +0.0431
  gamma 1.4                   0.8325    0.8234     +0.1014
  bias field +/-20%           0.9328    0.9234     +0.0014

  worst case across the suite: macro F1 0.4624 against a clean 0.9247
  saved -> outputs/robustness.png

Read the bias-field row against the noise and blur rows. Bias field is the most
MRI-specific corruption here and a model reading anatomy should barely notice
it; noise and blur disturb high-frequency texture. Which of those hurts more
says something about what the model is using.


In [9]:
# 8. OCCLUSION — WHAT THE MODEL NEEDS, NOT WHERE IT LOOKS
"""
A heatmap shows where gradient flows. It does not show what the prediction
depends on, and the two come apart: a map can sit on a region the classifier
would happily do without.

Occlusion answers the stronger question and needs no annotations. A patch slides
across the image, and the drop in the predicted class probability is recorded at
each position. Regions the model needs produce a large drop.

The patch is filled with the median brain intensity rather than black, because a
black square is a high-contrast object appearing in no training image, and the
drop it causes would partly measure surprise at the artefact rather than the
loss of information.
"""
fig, axes = viz.styled_fig(2, 3, figsize=(11, 7.5))
for ax_col, (c, name) in enumerate(zip(range(len(CLASSES)), CLASSES)):
    if ax_col >= 3:
        break
    idx = [i for i in np.where(y_true == c)[0] if y_pred[i] == c][:1]
    if not idx:
        continue
    i = idx[0]
    heat, cls, base_p = explain.occlusion_map(model, test_img[i], eval_tf)
    axes[0, ax_col].imshow(test_img[i], cmap='gray')
    axes[0, ax_col].set_title(f"{name}  p={base_p:.2f}", fontsize=9)
    axes[1, ax_col].imshow(test_img[i], cmap='gray')
    axes[1, ax_col].imshow(np.asarray(Image.fromarray((heat * 255).astype(np.uint8))
                                      .resize(test_img[i].shape[::-1])),
                           cmap='jet', alpha=0.55)
    axes[1, ax_col].set_title("occlusion sensitivity", fontsize=9)
for ax in axes.ravel():
    ax.axis('off')
plt.suptitle("What the prediction actually depends on", fontsize=12, fontweight='bold')
plt.tight_layout(); viz.save(fig, "occlusion.png")

print("Bright regions are those whose removal costs the model the most confidence.")
print("This is a perturbation of the input rather than an inspection of gradients,")
print("so it reports what the prediction depends on rather than where activation")
print("happens to be -- and those two are not the same thing.")

  saved -> outputs/occlusion.png
Bright regions are those whose removal costs the model the most confidence.
This is a perturbation of the input rather than an inspection of gradients,
so it reports what the prediction depends on rather than where activation
happens to be -- and those two are not the same thing.


In [10]:
# 9. SUMMARY AND HONEST LIMITATIONS
"""
The limitations are not boilerplate. Each is a specific reason the number above
could be optimistic, and naming them is what separates a result from a claim.
"""
macro_f1_v = f1
low_acc = correct[cos < 0.85].mean()
worst_rob = min(v["f1"] for k, v in rob.items() if k != "clean")

print("=" * 66)
print("PHASE 5 — RESULTS")
print("=" * 66)
print(f"  macro F1                     {macro_f1_v:.4f}   95% CI [{f1_lo:.4f}, {f1_hi:.4f}]")
print(f"  accuracy                     {acc:.4f}   95% CI [{acc_lo:.4f}, {acc_hi:.4f}]")
print(f"  macro AUC                    {macro_auc:.4f}")
print(f"  weakest class                {min(rows[:len(CLASSES)], key=lambda r: r['recall'])['class']}"
      f" (recall {min(r['recall'] for r in rows[:len(CLASSES)]):.4f})")
print(f"  accuracy on non-512x512      {correct[~is512].mean():.4f}   <- section 5")
print(f"  accuracy, least-similar band {low_acc:.4f}   <- section 6")
print(f"  worst case under corruption  {worst_rob:.4f} macro F1   <- section 7")

checks = [
    ("every held-out image scored",       len(y_true) == int(KEEP.sum())),
    ("leaked images excluded",            int(KEEP.sum()) < len(KEEP)),
    ("checkpoint matches run manifest",   ckpt["manifest"] == manifest.current_hash()),
    ("accuracy above chance",             acc > 0.5),
    ("accuracy not implausibly high",     acc <= 0.995),
    ("all classes have non-zero recall",  all(r["recall"] > 0 for r in rows[:len(CLASSES)])),
    ("figures written", all((config.OUTPUTS / f).exists() for f in
        ["confusion_matrix.png", "roc_curves.png", "shortcut_audit.png",
         "robustness.png", "occlusion.png"])),
]
print("\n  verification:")
for label, ok in checks:
    print(f"    {'OK  ' if ok else 'FAIL'}  {label}")
failed = [l for l, ok in checks if not ok]
assert not failed, "failed checks: " + "; ".join(failed)

print(f"""
  LIMITATIONS

  1. A metadata shortcut exists in this dataset and was measured before training
     rather than inferred afterwards: a classifier given only width, height,
     file size and compression -- no pixels -- reaches 0.6158 against 0.2500
     chance. Section 5 reports what this model did with it. The size-stratified
     figures belong beside the headline wherever it is quoted.

  2. No patient identifiers. Phase 1 removed every duplicate and every
     near-neighbour above cosine 0.95 and asserted the removal, but two
     different slices of one patient are not duplicates and no pixel test finds
     them. Section 6 bounds the residual; it cannot eliminate it.

  3. No tumour annotations. Section 8 measures which regions the prediction
     depends on, by perturbing the input and observing the consequence, but it
     cannot state that those regions are lesions -- there is no ground-truth
     lesion to compare against. Any claim that the maps land on tumours would
     be an impression rather than a result.

  4. The test set is small and uneven after cleaning, from 88 notumor images to
     375 glioma. That is a consequence of honest cleaning rather than a property
     of the disease, and it is why macro F1 leads and accuracy follows.

  5. Clean-data performance only. Section 7 degrades the test set with the
     variation that separates scanners; the accuracy above holds under this
     dataset's conditions, and the worst case there is the number any deployment
     claim must be made against.

  6. Single split, single seed. The intervals here capture sampling variation in
     the test set, not variation between training runs.

  7. Not a clinical result. Retrospective JPEGs of unknown provenance, scored
     offline, with no comparison against radiologist performance and no
     prospective validation.""")

PHASE 5 — RESULTS
  macro F1                     0.9247   95% CI [0.9053, 0.9431]
  accuracy                     0.9367   95% CI [0.9211, 0.9513]
  macro AUC                    0.9863
  weakest class                glioma (recall 0.8560)
  accuracy on non-512x512      0.7202   <- section 5
  accuracy, least-similar band 0.9005   <- section 6
  worst case under corruption  0.4624 macro F1   <- section 7

  verification:
    OK    every held-out image scored
    OK    leaked images excluded
    OK    checkpoint matches run manifest
    OK    accuracy above chance
    OK    accuracy not implausibly high
    OK    all classes have non-zero recall
    OK    figures written

  LIMITATIONS

  1. A metadata shortcut exists in this dataset and was measured before training
     rather than inferred afterwards: a classifier given only width, height,
     file size and compression -- no pixels -- reaches 0.6158 against 0.2500
     chance. Section 5 reports what this model did with it. The size-str